# Validar benchmark con imágenes reducidas

Este notebook comprueba si la reducción de resolución se aplica antes del benchmark y cómo cambia visualmente la imagen de entrada y sus anotaciones.

El flujo que sigue es:
1. Cargar una imagen representativa del dataset.
2. Reducirla con varios factores de escala.
3. Comparar imagen original vs. redimensionada.
4. Preparar una mini-validación para comparar métricas o lanzar el benchmark con distintos tamaños.

In [ ]:
import sys
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "pyproject.toml").exists()),
    Path.cwd(),
)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.pose_uncertainty.data_loader import COCOLoader
from src.pose_uncertainty.utils.types import ImageSample

plt.style.use("seaborn-v0_8-whitegrid")

print(f"Project root: {project_root}")
print("Notebook listo para probar la reducción de resolución.")

# Ejecutar el preprocesado antes del benchmark

Aquí se valida que la reducción de resolución no solo cambia el tamaño de la imagen, sino que también escala correctamente el bounding box y los keypoints asociados.

In [ ]:
dataset_root = project_root / "data" / "coco"
annotation_file = "annotations/person_keypoints_val2017.json"
image_subdir = "val2017"
benchmark_scales = [1.0, 0.75, 0.5, 0.25]

loader_args = {
    "data_root": str(dataset_root),
    "ann_file": annotation_file,
    "image_dir": image_subdir,
}

sample_variants = {
    scale: next(iter(COCOLoader(**loader_args, resize_scale=scale)))
    for scale in benchmark_scales
}

sample_full = sample_variants[1.0]

comparison_df = pd.DataFrame(
    [
        {
            "scale": scale,
            "image_width": sample.image_array.shape[1],
            "image_height": sample.image_array.shape[0],
            "bbox_x": round(sample.bbox[0], 2),
            "bbox_y": round(sample.bbox[1], 2),
            "bbox_w": round(sample.bbox[2], 2),
            "bbox_h": round(sample.bbox[3], 2),
        }
        for scale, sample in sample_variants.items()
    ]
).sort_values("scale", ascending=False)

display(comparison_df)
print(f"Imagen base: {sample_full.image_array.shape}")
print(f"BBox base: {tuple(round(v, 2) for v in sample_full.bbox)}")

In [ ]:
def resize_image_and_annotations(image, bbox, keypoints, scale):
    """Resize an image and scale bbox/keypoints with the same factor."""
    if scale is None or abs(scale - 1.0) < 1e-8:
        return image, bbox, keypoints

    if scale <= 0:
        raise ValueError(f"scale must be > 0, got {scale}")

    height, width = image.shape[:2]
    new_width = max(1, int(round(width * scale)))
    new_height = max(1, int(round(height * scale)))
    interpolation = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_LINEAR
    resized = cv2.resize(image, (new_width, new_height), interpolation=interpolation)

    x, y, w, h = bbox
    resized_bbox = (x * scale, y * scale, w * scale, h * scale)

    resized_keypoints = []
    for kp in keypoints:
        resized_keypoints.append(
            type(kp)(
                id=kp.id,
                x=kp.x * scale,
                y=kp.y * scale,
                confidence=kp.confidence,
                name=kp.name,
            )
        )

    return resized, resized_bbox, resized_keypoints


def build_scaled_sample(sample, scale):
    image, bbox, keypoints = resize_image_and_annotations(
        sample.image_array,
        sample.bbox,
        sample.ground_truth_keypoints or [],
        scale,
    )
    return ImageSample(
        image_id=sample.image_id,
        image_path=sample.image_path,
        image_array=image,
        bbox=bbox,
        ground_truth_keypoints=keypoints,
        dataset_source=sample.dataset_source,
    )


def draw_pose_sample(ax, image, bbox, keypoints, title):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")

    x, y, w, h = bbox
    ax.add_patch(
        plt.Rectangle(
            (x, y),
            w,
            h,
            fill=False,
            edgecolor="cyan",
            linewidth=2,
        )
    )

    for kp in keypoints:
        ax.scatter(kp.x, kp.y, s=18, c="yellow", edgecolors="black", linewidths=0.4)
        ax.text(kp.x + 1.5, kp.y + 1.5, kp.name, fontsize=6, color="white")

# Comparar imagen original vs. imagen reducida

En esta sección se ve el impacto directo de `resize_scale` sobre la imagen que entra al benchmark. La idea es comprobar que la reducción mantiene la estructura general de la escena y, al mismo tiempo, elimina detalle fino.

In [ ]:
fig, axes = plt.subplots(len(benchmark_scales), 2, figsize=(14, 4 * len(benchmark_scales)))
if len(benchmark_scales) == 1:
    axes = np.array([axes])

for row, scale in enumerate(benchmark_scales):
    resized_sample = sample_variants[scale]
    manual_sample = build_scaled_sample(sample_full, scale)

    draw_pose_sample(
        axes[row, 0],
        sample_full.image_array,
        sample_full.bbox,
        sample_full.ground_truth_keypoints or [],
        f"Original - {sample_full.image_array.shape[1]}x{sample_full.image_array.shape[0]}",
    )
    draw_pose_sample(
        axes[row, 1],
        resized_sample.image_array,
        resized_sample.bbox,
        resized_sample.ground_truth_keypoints or [],
        f"Loader resize_scale={scale} - {resized_sample.image_array.shape[1]}x{resized_sample.image_array.shape[0]}",
    )

    print(
        f"scale={scale}: loader shape={resized_sample.image_array.shape}, manual shape={manual_sample.image_array.shape}, "
        f"bbox_loader={tuple(round(v, 2) for v in resized_sample.bbox)}, bbox_manual={tuple(round(v, 2) for v in manual_sample.bbox)}"
    )

plt.tight_layout()
plt.show()

In [ ]:
benchmark_scales = [1.0, 0.75, 0.5, 0.25]
benchmark_commands = [
    f"python src/experiments/run_benchmark.py dataset.resize_scale={scale} force_rerun=true"
    for scale in benchmark_scales
]

print("Comandos de benchmark sugeridos:")
for command in benchmark_commands:
    print("  ", command)

RUN_MINI_BENCHMARK = False

if RUN_MINI_BENCHMARK:
    try:
        from omegaconf import OmegaConf
        from src.pose_uncertainty.evaluation.runner import EvaluationRunner

        cfg = OmegaConf.load(project_root / "configs" / "config.yaml")
        cfg.dataset.resize_scale = 1.0
        runner = EvaluationRunner(cfg)

        rows = []
        for scale in benchmark_scales:
            scaled_sample = build_scaled_sample(sample_full, scale)
            metrics = runner._evaluate_single(scaled_sample)
            if metrics is None:
                continue
            rows.append(
                {
                    "scale": scale,
                    "image_width": scaled_sample.image_array.shape[1],
                    "image_height": scaled_sample.image_array.shape[0],
                    "bbox": scaled_sample.bbox,
                    "oks_base": metrics.get("oks_base"),
                    "oks_ours": metrics.get("oks_ours"),
                    "delta_oks": metrics.get("delta_oks"),
                    "nll": metrics.get("nll"),
                    "entropy": metrics.get("entropy"),
                }
            )

        benchmark_df = pd.DataFrame(rows)
        display(benchmark_df)
    except Exception as exc:
        print(f"No se pudo ejecutar el mini-benchmark: {exc}")
else:
    print("Pon RUN_MINI_BENCHMARK = True para ejecutar una evaluación rápida de una sola imagen por escala.")

In [ ]:
output_dir = project_root / "outputs" / "notebooks" / "downscale_validation"
output_dir.mkdir(parents=True, exist_ok=True)

if "comparison_df" in globals():
    comparison_df.to_csv(output_dir / "scale_comparison.csv", index=False)
    print(f"Tabla guardada en: {output_dir / 'scale_comparison.csv'}")

if "benchmark_df" in globals():
    benchmark_df.to_csv(output_dir / "mini_benchmark_metrics.csv", index=False)
    print(f"Métricas guardadas en: {output_dir / 'mini_benchmark_metrics.csv'}")

if "fig" in globals():
    fig.savefig(output_dir / "scale_comparison.png", dpi=180, bbox_inches="tight")
    print(f"Figura guardada en: {output_dir / 'scale_comparison.png'}")

print(f"Directorio de salida listo: {output_dir}")

# Guardado y siguientes pasos

Los artefactos de esta validación se guardan en `outputs/notebooks/downscale_validation/`.

Para ejecutar el benchmark real con una resolución concreta, usa un override de Hydra como este:

```bash
python src/experiments/run_benchmark.py dataset.resize_scale=0.5 force_rerun=true
```

Si además quieres que `launch_viz.py` conserve artefactos visuales, evita combinar esa ejecución con `evaluation.debug_limit`, porque ese atajo consume el dataloader antes de la fase de deep profiling.